# Modul 10: Dasar-dasar Deteksi Obyek

Nama: Sahrul Ramadhani  
NIM: 244107020058  
Kelas: TI 3C

## D1. Praktikum Deteksi Obyek Dasar

### 1. Import library dan akses Google Drive

In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    versi_cv = importlib.metadata.version('opencv-python')
except importlib.metadata.PackageNotFoundError:
    versi_cv = '0'
if not versi_cv.startswith('4.'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python==4.12.0.88'])

import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path
import os

try:
    from google.colab import drive
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/PCVK/Minggu 10')
else:
    pilihan = [Path.cwd(), Path.cwd() / 'Minggu 10']
    BASE = Path(os.environ['PCVK_MINGGU10']) if os.environ.get('PCVK_MINGGU10') else next(
        (p for p in pilihan if (p / 'images').is_dir()), pilihan[-1]
    )

IMAGES = BASE / 'images'
HASIL = BASE / 'hasil'
HASIL.mkdir(parents=True, exist_ok=True)
plt.rcParams['figure.dpi'] = 100

def baca(nama, grayscale=False):
    path = IMAGES / nama
    image = cv.imread(str(path), cv.IMREAD_GRAYSCALE if grayscale else cv.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f'Gambar tidak ditemukan atau tidak dapat dibaca: {path}')
    return image

def tampilkan(images, titles, cols=None, figsize=None):
    cols = len(images) if cols is None else cols
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize or (6 * cols, 4.5 * rows), squeeze=False)
    for ax, image, title in zip(axes.flat, images, titles):
        if image.ndim == 2:
            ax.imshow(image, cmap='gray')
        else:
            ax.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    for ax in list(axes.flat)[len(images):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

print('Folder gambar:', IMAGES)

### 2. Enam metode template matching OpenCV

In [ ]:
source_cat = baca('Object Detection/cats_and_bunnies.jpg')
gray_cat = cv.cvtColor(source_cat, cv.COLOR_BGR2GRAY)
template_cat = baca('Object Detection/cat2_templatejpg.jpg', grayscale=True)
h, w = template_cat.shape
if h > gray_cat.shape[0] or w > gray_cat.shape[1]:
    raise ValueError('Ukuran template melebihi gambar sumber')
tampilkan([source_cat, template_cat], ['Gambar sumber', 'Template'])

methods = ['TM_CCOEFF', 'TM_CCOEFF_NORMED', 'TM_CCORR',
           'TM_CCORR_NORMED', 'TM_SQDIFF', 'TM_SQDIFF_NORMED']
hasil_template = {}
for nama in methods:
    method = getattr(cv, nama)
    response = cv.matchTemplate(gray_cat, template_cat, method)
    min_val, max_val, min_loc, max_loc = cv.minMaxLoc(response)
    top_left = min_loc if method in [cv.TM_SQDIFF, cv.TM_SQDIFF_NORMED] else max_loc
    bottom_right = (top_left[0] + w, top_left[1] + h)
    detected = source_cat.copy()
    cv.rectangle(detected, top_left, bottom_right, (0, 0, 255), 3)
    hasil_template[nama] = {'lokasi': top_left, 'response': response}
    tampilkan([response, detected], ['Hasil pencocokan: ' + nama, 'Obyek terdeteksi'])

### 3. Template matching manual untuk banyak obyek

In [ ]:
def jumlah_window(image, h, w):
    integral = np.pad(image, ((1, 0), (1, 0))).cumsum(0).cumsum(1)
    return integral[h:, w:] - integral[:-h, w:] - integral[h:, :-w] + integral[:-h, :-w]

def template_matching_manual(image, template):
    image = image.astype(np.float64)
    template = template.astype(np.float64)
    h, w = template.shape
    if h > image.shape[0] or w > image.shape[1]:
        raise ValueError('Ukuran template melebihi gambar sumber')
    template_zero = template - template.mean()
    shape = (image.shape[0] + h - 1, image.shape[1] + w - 1)
    correlation = np.fft.irfft2(
        np.fft.rfft2(image, s=shape) * np.fft.rfft2(template_zero[::-1, ::-1], s=shape), s=shape
    )[h - 1:image.shape[0], w - 1:image.shape[1]]
    total = jumlah_window(image, h, w)
    total2 = jumlah_window(image * image, h, w)
    denominator = np.sqrt(np.maximum(total2 - total * total / (h * w), 0) * np.sum(template_zero ** 2))
    return np.clip(np.divide(correlation, denominator, out=np.zeros_like(correlation),
                             where=denominator > 1e-10), -1, 1)

def pilih_puncak(response, template_shape, threshold=0.45):
    h, w = template_shape
    work = response.copy()
    peaks = []
    while True:
        y, x = np.unravel_index(np.argmax(work), work.shape)
        score = float(work[y, x])
        if score < threshold:
            break
        peaks.append((int(x), int(y), score))
        work[max(0, y - h // 2):y + h // 2 + 1, max(0, x - w // 2):x + w // 2 + 1] = -np.inf
    return peaks

bahrain = baca('Object Detection/bahrain.jpg')
bahrain_gray = cv.cvtColor(bahrain, cv.COLOR_BGR2GRAY)
bahrain_template = baca('Object Detection/bahrain-template.jpg', grayscale=True)
response_manual = template_matching_manual(bahrain_gray, bahrain_template)
puncak_bahrain = pilih_puncak(response_manual, bahrain_template.shape)
bahrain_detected = bahrain.copy()
h, w = bahrain_template.shape
for x, y, score in puncak_bahrain:
    cv.rectangle(bahrain_detected, (x, y), (x + w, y + h), (0, 255, 255), 2)
tampilkan([bahrain, bahrain_template, bahrain_detected],
          ['Gambar sumber', 'Template', 'Template matching manual'])
print('Jumlah obyek:', len(puncak_bahrain))

### 4. Sobel, Canny, dan Laplacian edge detection

In [ ]:
car_park = baca('Object Detection/car-park.jpg')
car_gray = cv.cvtColor(car_park, cv.COLOR_BGR2GRAY)
sobel_x = cv.Sobel(car_gray, cv.CV_64F, 1, 0, ksize=3)
sobel_y = cv.Sobel(car_gray, cv.CV_64F, 0, 1, ksize=3)
sobel = cv.convertScaleAbs(np.hypot(sobel_x, sobel_y))
car_blur = cv.GaussianBlur(car_gray, (3, 3), 0)
canny = cv.Canny(car_blur, 50, 150)
laplacian = cv.convertScaleAbs(cv.Laplacian(car_blur, cv.CV_64F, ksize=3))
tampilkan([car_park, sobel], ['Gambar sumber', 'Sobel edge detection'])
tampilkan([car_park, canny], ['Gambar sumber', 'Canny edge detection'])
tampilkan([car_park, laplacian], ['Gambar sumber', 'Laplacian edge detection'])

### 5. Harris dan Shi-Tomasi corner detection

In [ ]:
chess = baca('Object Detection/chess-board.jpg')
chess_gray = cv.cvtColor(chess, cv.COLOR_BGR2GRAY)
harris_response = cv.cornerHarris(np.float32(chess_gray), 2, 3, 0.04)
harris_response = cv.dilate(harris_response, None)
harris = chess.copy()
harris[harris_response > 0.01 * harris_response.max()] = [0, 0, 255]

corners = cv.goodFeaturesToTrack(chess_gray, maxCorners=100, qualityLevel=0.01, minDistance=10)
shi_tomasi = chess.copy()
if corners is not None:
    for x, y in np.rint(corners.reshape(-1, 2)).astype(int):
        cv.circle(shi_tomasi, (x, y), 4, (255, 0, 0), -1)
tampilkan([harris, shi_tomasi], ['Harris corner detection', 'Shi-Tomasi corner detection'])

### 6. Grid detection dengan Hough Transform

In [ ]:
def gabungkan_garis(lines, rho_tolerance=12, theta_tolerance=np.deg2rad(3)):
    groups = []
    if lines is None:
        return []
    for rho, theta in lines[:, 0]:
        if theta > np.pi / 2:
            rho, theta = -rho, theta - np.pi
        for group in groups:
            r0, t0 = np.mean(group, axis=0)
            if abs(rho - r0) < rho_tolerance and abs(theta - t0) < theta_tolerance:
                group.append((rho, theta))
                break
        else:
            groups.append([(rho, theta)])
    return [tuple(np.mean(group, axis=0)) for group in groups]

sudoku = baca('Object Detection/sudoku.jpg')
sudoku_gray = cv.cvtColor(sudoku, cv.COLOR_BGR2GRAY)
grid_canny = cv.Canny(sudoku_gray, 100, 200)
grid_dilated = cv.dilate(grid_canny, np.ones((3, 3), np.uint8), iterations=1)
grid_eroded = cv.erode(grid_dilated, np.ones((3, 3), np.uint8), iterations=1)
grid_lines = cv.HoughLines(grid_eroded, 1, np.pi / 180, 180)
grid_filtered = None
if grid_lines is not None:
    arah_utama = grid_lines[0, 0, 1]
    selisih = np.abs(grid_lines[:, 0, 1] - arah_utama)
    selisih = np.minimum(selisih, np.pi - selisih)
    sesuai_grid = (selisih < np.deg2rad(12)) | (np.abs(selisih - np.pi / 2) < np.deg2rad(12))
    grid_filtered = grid_lines[sesuai_grid]
grid_merged = gabungkan_garis(grid_filtered)
grid_result = sudoku.copy()
for rho, theta in grid_merged:
    a, b = np.cos(theta), np.sin(theta)
    x0, y0 = a * rho, b * rho
    pt1 = (int(round(x0 - 1000 * b)), int(round(y0 + 1000 * a)))
    pt2 = (int(round(x0 + 1000 * b)), int(round(y0 - 1000 * a)))
    cv.line(grid_result, pt1, pt2, (0, 0, 255), 2)
tampilkan([grid_canny, grid_dilated, grid_eroded, grid_result],
          ['Canny', 'Dilasi', 'Erosi', 'Hough Transform dan penggabungan garis'], cols=2)
print('Jumlah garis setelah digabung:', len(grid_merged))

### 7. Contour detection dengan findContours

In [ ]:
laptop = baca('Object Detection/laptop.jpg')
laptop_gray = cv.cvtColor(laptop, cv.COLOR_BGR2GRAY)
laptop_blur = cv.GaussianBlur(laptop_gray, (5, 5), 0)
laptop_edges = cv.Canny(laptop_blur, 30, 100)
contours, hierarchy = cv.findContours(laptop_edges, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)
laptop_result = laptop.copy()
cv.drawContours(laptop_result, contours, -1, (0, 0, 255), 2)
tampilkan([laptop, laptop_result], ['Gambar sumber', 'Contour detection'])

## D2. Praktikum Feature Matching dan Face Detection

### Feature matching dengan ORB

In [ ]:
reeses = baca('facedet/reeses_puffs.png', grayscale=True)
cereals = baca('facedet/many_cereals.jpg', grayscale=True)
tampilkan([reeses, cereals], ['Reeses Puffs', 'Cereals'])
orb = cv.ORB_create()
kp1, des1 = orb.detectAndCompute(reeses, None)
kp2, des2 = orb.detectAndCompute(cereals, None)
bf = cv.BFMatcher(cv.NORM_HAMMING, crossCheck=True)
matches_orb = bf.match(des1, des2)
matches_orb = sorted(matches_orb, key=lambda x: x.distance)
reeses_match = cv.drawMatches(reeses, kp1, cereals, kp2, matches_orb[:25], None, flags=2)
tampilkan([reeses_match], ['ORB: 25 kecocokan terbaik'], figsize=(14, 6))
print('Jumlah kecocokan ORB:', len(matches_orb))

### Feature matching dengan SIFT dan ratio test

In [ ]:
sift = cv.SIFT_create()
kp1_sift, des1_sift = sift.detectAndCompute(reeses, None)
kp2_sift, des2_sift = sift.detectAndCompute(cereals, None)
bf_sift = cv.BFMatcher()
matches_sift = bf_sift.knnMatch(des1_sift, des2_sift, k=2)
good_sift = []
for pair in matches_sift:
    if len(pair) == 2:
        match1, match2 = pair
        if match1.distance < 0.75 * match2.distance:
            good_sift.append([match1])
sift_matches = cv.drawMatchesKnn(reeses, kp1_sift, cereals, kp2_sift, good_sift, None, flags=2)
tampilkan([sift_matches], ['SIFT dan ratio test'], figsize=(14, 6))
print('Jumlah kecocokan SIFT:', len(good_sift))

### 8. Face detection

In [ ]:
def muat_cascade(nama):
    path_drive = IMAGES / 'haarcascades' / nama
    path = path_drive if path_drive.is_file() else Path(cv.data.haarcascades) / nama
    cascade = cv.CascadeClassifier(str(path))
    if cascade.empty():
        raise FileNotFoundError(f'File Haar cascade tidak dapat dibaca: {path}')
    return cascade

cascade_wajah = muat_cascade('haarcascade_frontalface_alt.xml')

def deteksi_wajah(image):
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    return cascade_wajah.detectMultiScale(gray, scaleFactor=1.05, minNeighbors=4, minSize=(20, 20))

def gambar_kotak(image, boxes, color=(0, 255, 0)):
    result = image.copy()
    for x, y, w, h in boxes:
        cv.rectangle(result, (x, y), (x + w, y + h), color, 2)
    return result

nama_wajah = ['jokowi.jpg', 'kartini.jpg', 'kucing.jpg', 'mask.png', 'mjordan.jpg', 'solvayconf.jpg']
hasil_wajah = {}
prabowo = baca('prabowo.jpg')
wajah_prabowo = deteksi_wajah(prabowo)
tampilkan([prabowo, gambar_kotak(prabowo, wajah_prabowo)], ['Prabowo', 'Face detection'])
for nama in nama_wajah:
    image = baca('facedet/' + nama)
    boxes = deteksi_wajah(image)
    hasil_wajah[nama] = boxes
    tampilkan([image, gambar_kotak(image, boxes)], [nama, 'Face detection'])
    print(nama, ':', len(boxes), 'wajah')

### 9. Deteksi wajah kucing

In [ ]:
cascade_kucing = muat_cascade('haarcascade_frontalcatface_extended.xml')
kucing = baca('facedet/kucing.jpg')
kucing_gray = cv.cvtColor(kucing, cv.COLOR_BGR2GRAY)
wajah_kucing = cascade_kucing.detectMultiScale(kucing_gray, scaleFactor=1.05, minNeighbors=3, minSize=(100, 100))
tampilkan([kucing, gambar_kotak(kucing, wajah_kucing)], ['Gambar sumber', 'Deteksi wajah kucing'])
print('Jumlah wajah kucing:', len(wajah_kucing))

### 10. Eyes detection pada wajah Pak Prabowo

In [ ]:
cascade_mata = muat_cascade('haarcascade_eye.xml')
prabowo_gray = cv.cvtColor(prabowo, cv.COLOR_BGR2GRAY)
mata_result = cv.cvtColor(prabowo_gray, cv.COLOR_GRAY2BGR)
mata_prabowo = []
for x, y, w, h in wajah_prabowo:
    roi = prabowo_gray[y:y + int(h * 0.65), x:x + w]
    eyes = cascade_mata.detectMultiScale(roi, scaleFactor=1.05, minNeighbors=3, minSize=(10, 10))
    for ex, ey, ew, eh in eyes:
        mata_prabowo.append((x + ex, y + ey, ew, eh))
        cv.rectangle(mata_result, (x + ex, y + ey), (x + ex + ew, y + ey + eh), (255, 255, 255), 1)
tampilkan([mata_result], ['Eyes detection pada Pak Prabowo'])
print('Jumlah mata:', len(mata_prabowo))

### 11. Deteksi senyuman pada gambar people

In [ ]:
cascade_senyum = muat_cascade('haarcascade_smile.xml')
people = baca('people.jpg')
people_gray = cv.cvtColor(people, cv.COLOR_BGR2GRAY)
people_result = people.copy()
senyum_people = []
for x, y, w, h in deteksi_wajah(people):
    roi = people_gray[y + h // 2:y + h, x:x + w]
    smiles = cascade_senyum.detectMultiScale(roi, scaleFactor=1.1, minNeighbors=25, minSize=(15, 8))
    for sx, sy, sw, sh in smiles:
        sy += h // 2
        senyum_people.append((x + sx, y + sy, sw, sh))
        cv.rectangle(people_result, (x + sx, y + sy), (x + sx + sw, y + sy + sh), (0, 255, 0), 2)
tampilkan([people, people_result], ['Gambar people', 'Deteksi senyuman'])
print('Jumlah senyuman terdeteksi:', len(senyum_people))

### 12. Face tracking menggunakan kamera Google Colab

In [ ]:
import base64
from IPython.display import display, Javascript, JSON

def lacak_frame(data_url):
    data = base64.b64decode(data_url.split(',', 1)[1])
    frame = cv.imdecode(np.frombuffer(data, dtype=np.uint8), cv.IMREAD_COLOR)
    if frame is None:
        raise ValueError('Frame kamera tidak dapat dibaca')
    boxes = deteksi_wajah(frame)
    result = gambar_kotak(frame, boxes)
    ok, encoded = cv.imencode('.jpg', result)
    if not ok:
        raise ValueError('Frame hasil tidak dapat disimpan')
    return JSON({'image': 'data:image/jpeg;base64,' + base64.b64encode(encoded).decode(),
                 'faces': len(boxes)})

KAMERA_JS = r"""
(() => {
  if (window.pcvk10Stop) window.pcvk10Stop();
  const panel = document.createElement('div');
  const start = document.createElement('button');
  const stop = document.createElement('button');
  const status = document.createElement('p');
  const video = document.createElement('video');
  const canvas = document.createElement('canvas');
  const result = document.createElement('img');
  start.textContent = 'Mulai kamera';
  stop.textContent = 'Hentikan kamera';
  stop.disabled = true;
  video.autoplay = true;
  video.muted = true;
  video.playsInline = true;
  video.style.display = 'none';
  result.style.maxWidth = '100%';
  status.textContent = 'Kamera belum dinyalakan';
  panel.append(start, stop, status, video, result);
  document.body.appendChild(panel);
  const resize = () => google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  result.onload = resize;
  resize();
  let stream = null;
  let running = false;
  const close = () => {
    running = false;
    if (stream) stream.getTracks().forEach(track => track.stop());
    stream = null;
    video.srcObject = null;
    start.disabled = false;
    stop.disabled = true;
    status.textContent = 'Kamera dihentikan';
  };
  window.pcvk10Stop = close;
  stop.onclick = close;
  window.addEventListener('pagehide', close, {once: true});
  start.onclick = async () => {
    start.disabled = true;
    try {
      stream = await navigator.mediaDevices.getUserMedia({video: true, audio: false});
      video.srcObject = stream;
      await video.play();
      canvas.width = 640;
      canvas.height = Math.round(640 * video.videoHeight / video.videoWidth);
      running = true;
      stop.disabled = false;
      while (running) {
        canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);
        const reply = await google.colab.kernel.invokeFunction(
          'pcvk10.face', [canvas.toDataURL('image/jpeg', 0.8)], {});
        if (!running) break;
        const payload = reply.data['application/json'];
        if (!payload || !payload.image) throw new Error('Frame hasil tidak tersedia');
        result.src = payload.image;
        status.textContent = 'Wajah terdeteksi: ' + payload.faces;
        await new Promise(resolve => setTimeout(resolve, 100));
      }
    } catch (error) {
      close();
      status.textContent = 'Kamera tidak dapat digunakan: ' + error.message;
    }
  };
})();
"""

if DI_COLAB:
    from google.colab import output
    output.register_callback('pcvk10.face', lacak_frame)
    display(Javascript(KAMERA_JS))
else:
    print('Kamera dijalankan melalui Google Colab')

### 13. Blurring pada wajah yang terdeteksi

In [ ]:
mjordan = baca('facedet/mjordan.jpg')
blur_result = mjordan.copy()
blur_boxes = deteksi_wajah(mjordan)
for x, y, w, h in blur_boxes:
    roi = blur_result[y:y + h, x:x + w]
    ukuran = min(99, min(roi.shape[:2]) // 2 * 2 - 1)
    if ukuran >= 3:
        blur_result[y:y + h, x:x + w] = cv.medianBlur(roi, ukuran)
tampilkan([mjordan, blur_result], ['Gambar sumber', 'Blurring wajah dengan medianBlur'])

### 14. Mengolah gambar KTM

#### a. Menampilkan KTM

In [ ]:
ktm = baca('ktm.jpg')
tampilkan([ktm], ['KTM'])

#### b. Menampilkan foto KTM

In [ ]:
wajah_ktm = deteksi_wajah(ktm)
if len(wajah_ktm) == 0:
    raise ValueError('Wajah pada KTM belum terdeteksi. Gunakan foto KTM yang tegak, jelas, dan tidak silau')
x, y, w, h = max(wajah_ktm, key=lambda box: int(box[2]) * int(box[3]))
x1 = max(0, x - int(0.4 * w))
y1 = max(0, y - int(0.55 * h))
x2 = min(ktm.shape[1], x + int(1.4 * w))
y2 = min(ktm.shape[0], y + int(2.25 * h))
foto_ktm = ktm[y1:y2, x1:x2]
tampilkan([foto_ktm], ['Foto KTM'])

#### c. Deteksi obyek KTM dengan feature matching

In [ ]:
ktm_scene = baca('ktm_scene.jpg')
ktm_gray = cv.cvtColor(ktm, cv.COLOR_BGR2GRAY)
scene_gray = cv.cvtColor(ktm_scene, cv.COLOR_BGR2GRAY)
sift_ktm = cv.SIFT_create()
kp_ktm, des_ktm = sift_ktm.detectAndCompute(ktm_gray, None)
kp_scene, des_scene = sift_ktm.detectAndCompute(scene_gray, None)
good_ktm = []
if des_ktm is not None and des_scene is not None and len(des_scene) >= 2:
    matches_ktm = cv.BFMatcher().knnMatch(des_ktm, des_scene, k=2)
    for pair in matches_ktm:
        if len(pair) == 2 and pair[0].distance < 0.75 * pair[1].distance:
            good_ktm.append(pair[0])

ktm_detected = ktm_scene.copy()
H_ktm = None
mask_ktm = None
proyeksi_ktm = None
if len(good_ktm) >= 4:
    src_pts = np.float32([kp_ktm[m.queryIdx].pt for m in good_ktm]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_scene[m.trainIdx].pt for m in good_ktm]).reshape(-1, 1, 2)
    H_ktm, mask_ktm = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
    if H_ktm is not None and int(mask_ktm.sum()) >= 4:
        h, w = ktm_gray.shape
        batas = np.float32([[0, 0], [0, h - 1], [w - 1, h - 1], [w - 1, 0]]).reshape(-1, 1, 2)
        proyeksi = cv.perspectiveTransform(batas, H_ktm)
        if np.isfinite(proyeksi).all() and cv.isContourConvex(proyeksi.astype(np.float32)):
            proyeksi_ktm = proyeksi
            cv.polylines(ktm_detected, [np.rint(proyeksi).astype(np.int32)], True, (0, 255, 0), 3, cv.LINE_AA)
if proyeksi_ktm is None:
    print('KTM belum terdeteksi. Gunakan dua foto kartu yang sama, jelas, dan tidak tertutup')

matches_mask = mask_ktm.ravel().tolist() if mask_ktm is not None else [0] * len(good_ktm)
ktm_feature_result = cv.drawMatches(ktm, kp_ktm, ktm_detected, kp_scene, good_ktm, None,
                                   matchesMask=matches_mask, flags=2)
tampilkan([ktm_feature_result], ['Feature matching KTM'], figsize=(16, 7))
tampilkan([ktm_detected], ['Deteksi obyek KTM'])
print('Jumlah kecocokan:', len(good_ktm))
print('Jumlah inlier:', int(mask_ktm.sum()) if mask_ktm is not None else 0)

## D3. Tugas Pengayaan: Pengenalan Karakter KTM dengan CNN

### Membuat folder dataset

In [ ]:
DATASET = BASE / 'dataset'
for angka in range(10):
    (DATASET / 'training' / str(angka)).mkdir(parents=True, exist_ok=True)
(DATASET / 'testing' / '244107020058').mkdir(parents=True, exist_ok=True)
print('Folder dataset:', DATASET)

### Persiapan data training

In [ ]:
import os
import cv2
import random
import numpy as np
import pickle
from tqdm import tqdm

DATADIR = BASE / 'dataset' / 'training'
width, height = 100, 100
if not DATADIR.is_dir():
    raise FileNotFoundError(f'Folder training belum tersedia: {DATADIR}')
dirs = sorted(p.name for p in DATADIR.iterdir() if p.is_dir() and not p.name.startswith('.'))
angka_belum_ada = [str(i) for i in range(10) if str(i) not in dirs]
if angka_belum_ada:
    raise ValueError('Folder kelas angka belum lengkap: ' + ', '.join(angka_belum_ada))
training_data = []
for char_name in dirs:
    path = DATADIR / char_name
    class_number = dirs.index(char_name)
    files = sorted(p for p in path.iterdir() if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'])
    if not files:
        raise ValueError(f'Folder kelas kosong: {path}')
    for img in tqdm(files, desc=char_name):
        img_array = cv2.imread(str(img), cv2.IMREAD_GRAYSCALE)
        if img_array is None:
            raise ValueError(f'Gambar training tidak dapat dibaca: {img}')
        new_array = cv2.resize(img_array, (width, height))
        training_data.append([new_array, class_number])

random.shuffle(training_data)
X, Y = [], []
for feature, label in training_data:
    X.append(feature)
    Y.append(label)
X = np.array(X).reshape(-1, width, height, 1)
Y = np.array(Y)
for nama, data in [('X.pickle', X), ('Y.pickle', Y), ('kelas.pickle', dirs)]:
    with open(HASIL / nama, 'wb') as file:
        pickle.dump(data, file)
print('jumlah data :', len(training_data))
print('jumlah kelas:', len(dirs), dirs)

### Training data

In [ ]:
import pickle
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, ZeroPadding2D
from keras.models import Model
from keras.optimizers import Adam
from keras.utils import to_categorical

with open(HASIL / 'X.pickle', 'rb') as file:
    X = pickle.load(file)
with open(HASIL / 'Y.pickle', 'rb') as file:
    Y = pickle.load(file)
Y = to_categorical(Y)
X = X / 255.0
width, height = 100, 100
jumlah_kelas = Y.shape[1]

inputs = Input(shape=(width, height, 1))
conv_layer = ZeroPadding2D(padding=(2, 2))(inputs)
conv_layer = Conv2D(16, (5, 5), strides=(1, 1), activation='relu')(conv_layer)
conv_layer = MaxPooling2D((2, 2))(conv_layer)
conv_layer = Conv2D(32, (3, 3), strides=(1, 1), activation='relu')(conv_layer)
conv_layer = Conv2D(32, (3, 3), strides=(1, 1), activation='relu')(conv_layer)
conv_layer = MaxPooling2D((2, 2))(conv_layer)
conv_layer = Conv2D(64, (3, 3), strides=(1, 1), activation='relu')(conv_layer)
flaten = Flatten()(conv_layer)
fc_layer = Dense(256, activation='relu')(flaten)
fc_layer = Dense(64, activation='relu')(fc_layer)
outputs = Dense(jumlah_kelas, activation='softmax')(fc_layer)
model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
riwayat = model.fit(X, Y, epochs=20, verbose=1, validation_split=0.2)
model.save(HASIL / 'anpr.keras')
print('akurasi training  :', riwayat.history['accuracy'][-1])
print('akurasi validasi  :', riwayat.history['val_accuracy'][-1])

### Testing data

In [ ]:
import os
import cv2
import pickle
import numpy as np
import tensorflow as tf

data_dir_testing = BASE / 'dataset' / 'testing'
width, height = 100, 100
if not data_dir_testing.is_dir():
    raise FileNotFoundError(f'Folder testing belum tersedia: {data_dir_testing}')
folders_testing = sorted(p for p in data_dir_testing.iterdir() if p.is_dir() and not p.name.startswith('.'))
if not folders_testing:
    raise ValueError('Folder testing belum berisi folder potongan karakter KTM')
model = tf.keras.models.load_model(HASIL / 'anpr.keras')
with open(HASIL / 'kelas.pickle', 'rb') as file:
    dirs = pickle.load(file)
hasil_testing = {}
for folder in folders_testing:
    temp = ''
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'])
    if not files:
        raise ValueError(f'Folder testing kosong: {folder}')
    for char_img in files:
        img_array = cv2.imread(str(char_img), cv2.IMREAD_GRAYSCALE)
        if img_array is None:
            raise ValueError(f'Gambar testing tidak dapat dibaca: {char_img}')
        new_array = cv2.resize(img_array, (width, height))
        new_array = new_array.reshape(-1, width, height, 1) / 255.0
        prediction = model.predict(new_array, verbose=0)
        temp += dirs[np.argmax(prediction[0])]
    hasil_testing[folder.name] = temp
    print('folder name: {} no: {}'.format(folder.name, temp))